# MIVMIR embeddings vizualisation

- [ ] Find an appropriate case to look into:
    - [ ] `normalwarthog`, mivmir rank 0  
          `/opt/home/tor.bjorgen/repos/cg/mivmir_validation/work/70/859ab4952473d825a89f61a8653c8e/normalwarthog_compound-predictions.vcf` n=673
    - [ ] `enormousdingo`, rank 0  
          `/opt/home/tor.bjorgen/repos/cg/mivmir_validation/work/bf/e466392c63fc3e5afecd78a8cd939f/enormousdingo_compound-predictions.vcf`, n=3497
    - [ ] `pleasedsculpin`, mivmir rank 6
- [ ] Identify pathogenic variant, to use this as plot marker 
- [ ] Create an model that outputs MIVMIR embeddings in addition to score
- [ ] Run model inference on VCF, store to pd.Dataframe
    - [ ] model score
    - [ ] embeddings
    - [ ] explanation model output
- [ ] Visualze as network plot
- [ ] Allow tuning network plot using model explanations (showcases explainability)

In [ ]:
%%script bash
python3 -m pip list

In [ ]:
%%writefile embeddings_model.py

import numpy as np
import pandas as pd
import os
from glob import glob
from progressbar import ProgressBar

from rdds.variant_rank_score.model.model import VariantRankScoreModel
from rdds.lib.vcf import VCFReader, ParsableVariant

model = VariantRankScoreModel()
model.load_saved_model()

print(model._keras_model.layers)

layers = model._keras_model.layers
for layer in layers:
    print(layer.name, layer.input, layer.output)

output_layer = model._keras_model.get_layer('Confidences')
embeddings = output_layer.input
print(f"embeddings tensor: {embeddings}")

import tensorflow as tf
embeddings_model = tf.keras.Model(inputs=model._keras_model.input, outputs=[model._keras_model.output, embeddings])
embeddings_model.compile()
embeddings_model.summary(line_length=160)

VCF_FILES = [
    '/rdds/src/rdds/variant_rank_score/inference_exploration/mivmir_nextflow/embeddings-viz-data/normalwarthog_compound-predictions.vcf',
]

def _infer_pathogenicity_score(tensor_dict):
    output = embeddings_model(tensor_dict)
    score_classes = output[0]
    embeddings = output[1]
    score_classes = score_classes.numpy()
    prediction_class_pathogenic = score_classes[:, 0]
    embeddings = embeddings.numpy()[0, :]

    data = {'score': prediction_class_pathogenic}
    for i, embedding in enumerate(embeddings):
        data.update({f"e{i}": embedding})

    return pd.DataFrame(data)
    

# HACK: Copy of method from VRS model.py
def score_variant(variants,
                  model_input_spec,
                  explain_variant_score_threshold: float = 0.9,
                  ignore_clinvar_uncertain_conflicting_annotations: bool = True):
    def get_str_feature(variant: ParsableVariant,
                        name: str) -> bytes:
        if name in variant.parsed_fields:
            return variant.__getattribute__(name)
        else:
            return b''

    def get_num_feature(variant: ParsableVariant,
                        name: str) -> float:
        if name in variant.parsed_fields:
            value = variant.__getattribute__(name)
            if isinstance(value, (bytes, str)) and len(value) == 0:
                pass
            else:
                return value
        return 0.0

    #model_input_spec = self._generate_dataset_tensor_signature()
    model_input_data_spec, _ = model_input_spec  # Drop labels
    input_dict: Dict[str, Union[List, tf.Tensor]] = {}
    for tensor_spec in model_input_data_spec:
        input_dict.update({tensor_spec.name: []})
        for variant in variants:
            if tensor_spec.dtype == tf.string:
                str_data = get_str_feature(variant=variant, name=tensor_spec.name)
                if ignore_clinvar_uncertain_conflicting_annotations:
                    try:
                        if 'CLINVAR' in tensor_spec.name:  # TODO: Issue 257
                            clinvar_clnsig = str(variant.__getattribute__('CSQ_CLINVAR_CLNSIG')).lower()
                            if 'uncertain' in clinvar_clnsig or 'conflicting' in clinvar_clnsig:
                                str_data = b''
                    except AttributeError:
                        pass
                input_dict[tensor_spec.name].append(str_data)
            elif tensor_spec.dtype == tf.float32:
                input_dict[tensor_spec.name].append(get_num_feature(variant=variant, name=tensor_spec.name))
            else:
                raise ValueError(f'Unmapped input data dtype spec: {tensor_spec}')
        # Convert to Tensor
        tensor_data = input_dict[tensor_spec.name].copy()
        input_dict[tensor_spec.name] = tf.constant(value=tensor_data,
                                                   dtype=tensor_spec.dtype,
                                                   name=tensor_spec.name)
    return _infer_pathogenicity_score(tensor_dict=input_dict)
    ###
    if len(pathogenicity_scores) != len(variants):
        raise ValueError(f'Expected same amount of predictions as input data')
    idx_scores_above_threshold = np.flatnonzero(pathogenicity_scores >= explain_variant_score_threshold)
    df_dict = {}
    for tensor_name, tensor in input_dict.items():
        df_dict.update({tensor_name: tensor.numpy()[idx_scores_above_threshold]})
    df_selected_variants_for_explanation = pd.DataFrame.from_dict(df_dict)
    explanations_full = np.empty(shape=(len(variants), len(df_selected_variants_for_explanation.columns)))
    explanations_full.fill(np.nan)
    if len(idx_scores_above_threshold) > 0:
        explanations_full[idx_scores_above_threshold, :] = self._model_explainer.shap_values(X=df_selected_variants_for_explanation.values,
                                                                                             gc_collect=True)  # FIXME: Method call not supposed to be erroneous by typechecker
    explanations_df = pd.DataFrame(data=explanations_full, columns=self._features)
    result_df = pd.concat(objs=(pd.Series(pathogenicity_scores, name='pathogenicity_score'),
                                explanations_df),
                          axis=1)
    return result_df

def _infer_store_to_df(vcf_file_path: str):
    vcf_reader = VCFReader(vcf_file_path)

    total_df = None
    pbar = ProgressBar(max_value=vcf_reader.number_of_variants)
    pbar.start()
    for i, variant in enumerate(vcf_reader):
        parsed_variant = ParsableVariant(variant=variant, vep_csq_description=vcf_reader.csq_description)
        df = score_variant([parsed_variant], model_input_spec=model._generate_dataset_tensor_signature())
        df_default = model.score_variant([parsed_variant], explain_variant_score_threshold=0.0)  # TODO: SET NEW EXPL THR
        df = pd.concat((df, df_default), axis=1)
        df.index = [variant.ID]

        if total_df is None:
            total_df = df
        else:
            total_df = pd.concat((total_df, df), axis=0)
        pbar.increment(1)
    pbar.finish()

    csv_file_name = vcf_file_path.replace('.vcf', '.csv')
    assert csv_file_name != vcf_file_path
    print(f"Storing to: {csv_file_name}")
    total_df.to_csv(csv_file_name)

for vcf_file in VCF_FILES:
    _infer_store_to_df(vcf_file)


In [ ]:
# Create model
!bash -c ". /opt/pyenv/bin/activate && PYTHONPATH=/rdds/src python3 /rdds/src/rdds/variant_rank_score/inference_exploration/mivmir_nextflow/embeddings_model.py"

In [ ]:
import pandas as pd
df = pd.read_csv('/rdds/src/rdds/variant_rank_score/inference_exploration/mivmir_nextflow/embeddings-viz-data/normalwarthog_compound-predictions.csv',
                index_col=0)
df = df.reset_index()

# Compute similarity metric for network plot
- [ ] Embedding similarity
      Colmnwise, take the dot product accross all rows, you get a n-by-n matrix.
- [ ] Explanation model similarity

In [ ]:
embedding_cols = [name for name in df.columns if name.startswith('e')]
embeddings = df[embedding_cols]
embeddings

In [ ]:
import numpy as np
link_source = []
link_target = []
link_strength = []

for id_outer, row_outer in embeddings.iterrows():
    for id_inner, row_inner in embeddings.iterrows():
        # Having a connection is required to visualize all variants, skipping "self" drops isolated data points!
        #if id_outer == id_inner:
        #    continue
        # Take the dot product accross all embedding dimensions
        similarity_metric = np.dot(row_outer.values, row_inner.values)
        link_source.append(id_outer)
        link_target.append(id_inner)
        link_strength.append(similarity_metric)

link_strength = np.asarray(link_strength) / np.max(np.asarray(link_strength))
link_strength = link_strength.tolist()

#too_weak_idx = []
#for i, ls in enumerate(link_strength):
#    if ls <100:
#        too_weak_idx.append(i)

#for i in too_weak_idx:
#    del link_source[i]
#    del link_target[i]
#    del link_strength[i]

links_embeddings = pd.DataFrame({
    'sources': link_source,
    'targets': link_target,
    'link_strengths': link_strength
})
links_embeddings

#links_embeddings = links_embeddings[links_embeddings.link_strengths > 0.1]  # VIEW SIMILARITY
#links_embeddings = links_embeddings[links_embeddings.link_strengths < 0.01]  # VIEW DISSIMILARITY
# TODO: This groups similar points, we want to visualize dissimilarity!
links_embeddings, len(links_embeddings)

In [ ]:
points = df[['score', 'index']]

In [ ]:
points[points.score > 0.8]

In [ ]:
for idx, row in links_embeddings.iterrows():
    if idx == 263:
        print(row)

In [ ]:
from cosmograph import cosmo
# https://cosmos.gl/?path=/docs/api-reference--docs
# https://github.com/cosmograph-org/py_cosmograph/blob/main/js/config-props.json
# https://github.com/cosmograph-org/py_cosmograph/blob/main/cosmograph/base.py#L126
# https://github.com/cosmograph-org/py_cosmograph/blob/main/cosmograph/widget/__init__.py#L26

In [ ]:
def setup_ui_handler(plot):
    """
    Cosmos UI handler for interacting with points
    """
    def ui_handler(event: dict):
        if event['name'] != 'clicked_point_index':
            return
        point_index = event['new']
        print(point_index)
        plot.select_point_by_index([point_index])
        plot.focus_point_by_index(point_index)
    
    plot.observe(ui_handler)

In [ ]:
plot = cosmo(
    points = points,
    point_label_by = 'index',
    point_color_by = 'score',
    point_size=1,
    point_size_scale=0.1,
    links = links_embeddings,
    link_source_by = 'sources',
    link_target_by = 'targets',
    link_strength_by = 'link_strengths',
    simulation_repulsion=0.14,  # 0.13 and spring 1.0 interesting
    simulation_link_spring=2.0,  # cluster similar points
    #simulation_link_distance=10,  # Affects initial positioning of linked nodes
    show_dynamic_labels = True,
    show_labels = False,
    render_links = False,
    #simulation_gravity = 10.0,
    simulation_center = 0.8,
    random_seed = 0,
    point_sampling_distance=2,
    simulation_decay=1000,  # Smaller is faster
    simulation_friction=0.8,
   #link_arrows = True
    disable_simulation=False
)
plot

In [ ]:
setup_ui_handler(plot)

In [ ]:
plot.focus_point_by_index([263])
plot.focus_point_by_index(263)  # Causative variant
plot.fit_view_by_ids([263])

# Make use of explanations in clustering data

Cluster based on explanations.

Should one use the score as link?
Start of by setting links by score, then 

Do this by:
1. Setting cluster names
2. Setting new links

Would be cool to modify this on the fly.

PolyPhen seems promising.

In [ ]:
explanation_names = ['CSQ_PolyPhen', 'CSQ_SIFT',
       'CSQ_CLINVAR_CLNREVSTAT', 'CSQ_CLINVAR_CLNSIG',
       'most_severe_consequence', 'CSQ_MaxEntScan_alt', 'CSQ_MaxEntScan_diff',
       'CSQ_MES-SWA_acceptor_alt', 'CSQ_MES-SWA_donor_alt',
       'CSQ_MES-SWA_donor_diff', 'CSQ_SpliceAI_pred_DS_AL',
       'CSQ_SpliceAI_pred_DS_DG', 'CSQ_SpliceAI_pred_DS_DL', 'CSQ_REVEL_score',
       'CSQ_LoFtool', 'CSQ_GERP++_RS', 'CSQ_phastCons100way_vertebrate',
       'CSQ_phyloP100way_vertebrate', 'CADD', 'SWEGENAF', 'GNOMADAF_popmax',
       'CSQ_SpliceAI_pred_DS_AG', 'Frq']
explanations = df[explanation_names]
explanations

In [ ]:
link_src_expl = []
link_target_expl = []
link_strength_expl = []
for idx_outer, row_outer in explanations.iterrows():
    for idx_inner, row_inner in explanations.iterrows():
        similarity_metric = np.dot(row_outer, row_inner)
        if similarity_metric <= 0:
            continue
        link_src_expl.append(idx_outer)
        link_target_expl.append(idx_inner)
        link_strength_expl.append(similarity_metric)

link_strength_expl = np.asarray(link_strength_expl) / np.max(np.asarray(link_strength_expl))
link_strength_expl = link_strength_expl.tolist()

#too_weak_idx = []
#for i, ls in enumerate(link_strength):
#    if ls <100:
#        too_weak_idx.append(i)

#for i in too_weak_idx:
#    del link_source[i]
#    del link_target[i]
#    del link_strength[i]

links_explanations = pd.DataFrame({
    'sources': link_src_expl,
    'targets': link_target_expl,
    'link_strengths': link_strength_expl
})
links_explanations
        

In [ ]:
# View simulation based on all explanations
plot_explanations = cosmo(
    points = points,
    point_label_by = 'index',
    point_color_by = 'score',
    point_size=5,
    point_size_scale=0.1,
    links = links_explanations,
    link_source_by = 'sources',
    link_target_by = 'targets',
    link_strength_by = 'link_strengths',
    simulation_repulsion=10,  # Make sure unconnected points are pushed away
    simulation_link_spring=0.01,  # cluster similar points
    #simulation_link_distance=10,  # Affects initial positioning of linked nodes
    show_dynamic_labels = True,
    show_labels = False,
    render_links = False,
    #simulation_gravity = 10.0,
    #simulation_center = 2,
    random_seed = 0,
    point_sampling_distance=2,
    simulation_decay=1000,  # Smaller is faster
    simulation_friction=0.1,
   #link_arrows = True
    disable_simulation=False
)
plot_explanations

In [ ]:
plot_explanations.focus_point([263])
plot_explanations.focus_point(263)
plot_explanations.fit_view_by_ids([263])

In [ ]:
# Alternative take on links, based on feature attrbution
link_by_individual_explanation = None
for column in explanations.columns:
    links = []
    src = []
    trg = []
    for idx_outer, row_outer in explanations.iterrows():
        for idx_inner, row_inner in explanations.iterrows():
            similarity_metric = np.dot(row_outer[column], row_inner[column])
            if similarity_metric <= 0:
                continue
            src.append(idx_outer)
            trg.append(idx_inner)
            links.append(similarity_metric)
    _df = pd.DataFrame({
    'src': src,
    'trg': trg,
    f'link_{column}': links
    })
    if link_by_individual_explanation is None:
        link_by_individual_explanation = _df
    else:
        link_by_individual_explanation = pd.concat((link_by_individual_explanation, _df), axis=0)
link_by_individual_explanation

In [ ]:
link_by_individual_explanation[['src', 'trg', 'link_CSQ_PolyPhen']].dropna()

In [ ]:
plot_explanations = cosmo(
    points = points,
    point_label_by = 'index',
    point_color_by = 'score',
    point_size=5,
    point_size_scale=1,
    links = link_by_individual_explanation[['src', 'trg', 'link_CSQ_PolyPhen']].dropna(),
    link_source_by = 'src',
    link_target_by = 'trg',
    link_strength_by = 'link_CSQ_PolyPhen',
    #simulation_repulsion=3,  # 0.13 and spring 1.0 interesting
    simulation_link_spring=1.0,  # cluster similar points
    #simulation_link_distance=10,  # Affects initial positioning of linked nodes
    show_dynamic_labels = True,
    show_labels = False,
    render_links = False,
    #simulation_gravity = 10.0,
    #simulation_center = 2,
    random_seed = 0,
    point_sampling_distance=2,
    simulation_decay=1000,  # Smaller is faster
    simulation_friction=0.1,
   #link_arrows = True
    disable_simulation=False
)
plot_explanations

In [ ]:
plot_explanations.focus_point([263])
plot_explanations.focus_point(263)
plot_explanations.fit_view_by_ids([263])

In [ ]:
g = cosmo(
    pd.concat((explanations, df.score), axis=1),
    #point_id_by='lemma',
    #point_label_by='word',
    point_x_by='CSQ_PolyPhen',
    point_y_by='CSQ_SIFT',
    point_color_by='score',
    #point_size_by='frequency',
    point_size_scale=2,  # often have to play with this number to get the size right
    disable_point_size_legend=True
)
g

# Playing around with embedding and explanations

In [ ]:
links_embeddings_explanations = pd.concat((links_embeddings, links_explanations), axis=0)
links_embeddings_explanations

In [ ]:
mixed_embedding_explanation_points = df[explanation_names + ['score', 'index']]
mixed_embedding_explanation_points

In [ ]:
colname  = 'CSQ_PolyPhen'
#colname = 'CSQ_CLINVAR_CLNSIG'
mixed_embedding_explanation_points['cluster_id'] = 0
mixed_embedding_explanation_points.loc[mixed_embedding_explanation_points[colname] > mixed_embedding_explanation_points[colname].mean(), 'cluster_id'] = 1
mixed_embedding_explanation_points['cluster_strength'] = mixed_embedding_explanation_points.CSQ_PolyPhen * 10000
mixed_embedding_explanation_points.cluster_id.unique(), mixed_embedding_explanation_points['cluster_strength']

In [ ]:
plot = cosmo(
    points = mixed_embedding_explanation_points,
    point_label_by = 'index',
    point_color_by = 'score',
    #point_cluster_by = 'cluster_id',
    #point_cluster_strength_by = 'cluster_strength',
    #point_include_columns=['CSQ_CLINVAR_CLNREVSTAT'],
    point_size=10,
    point_size_scale=0.1,
    links = links_embeddings_explanations,
    #links = links_embeddings,
    link_source_by = 'sources',
    link_target_by = 'targets',
    link_strength_by = 'link_strengths',
    simulation_repulsion=20, # Make sure unconnected points are pushed away, 0.14 good
    simulation_link_spring=0.01,  # cluster similar points
    # repulsion 1 and spring 0.01 is good
    #simulation_link_distance=10,  # Affects initial positioning of linked nodes
    show_dynamic_labels = True,
    show_labels = True,
    render_links = False,
    #simulation_gravity = 10.0,
    simulation_center = 0.8,
    random_seed = 0,
    simulation_decay=1000,  # Smaller is faster
    simulation_friction=0.8,
   #link_arrows = True
    disable_simulation=False
)
plot

In [ ]:
setup_ui_handler(plot)

In [ ]:
# Causative variant
plot.focus_point([263])
plot.focus_point(263)
plot.fit_view_by_ids([263])

# Dimensionality Reduction of Embeddings
- https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html#sklearn.decomposition.PCA
- https://umap-learn.readthedocs.io/en/latest/index.html

In [ ]:
%%script bash
pip3 install scikit-learn umap-learn

In [ ]:
from sklearn.decomposition import PCA
from umap import UMAP

In [ ]:
pca = PCA(n_components=2)
embeddings_pca = pca.fit_transform(X=embeddings)
embeddings_pca = pd.DataFrame({'pca0': embeddings_pca[:, 0], 'pca1': embeddings_pca[:, 1]})
embeddings_pca = pd.concat((points, embeddings_pca), axis=1)
embeddings_pca

In [ ]:
plot_pca = cosmo(
    points = embeddings_pca,
    point_label_by = 'index',
    point_color_by = 'score',
    point_x_by = 'pca0',
    point_y_by = 'pca1',
    #point_cluster_by = 'cluster_id',
    #point_cluster_strength_by = 'cluster_strength',
    #point_include_columns=['CSQ_CLINVAR_CLNREVSTAT'],
    point_size=10,
    point_size_scale=1,
    simulation_repulsion=20, # Make sure unconnected points are pushed away, 0.14 good
    #simulation_link_spring=0.01,  # cluster similar points
    # repulsion 1 and spring 0.01 is good
    #simulation_link_distance=10,  # Affects initial positioning of linked nodes
    show_dynamic_labels = True,
    show_labels = True,
    render_links = False,
    #simulation_gravity = 10.0,
    #simulation_center = 0.8,
    random_seed = 0,
    simulation_decay=1000,  # Smaller is faster
    simulation_friction=0.8,
   #link_arrows = True
    disable_simulation=True
)
plot_pca

In [ ]:
# Causative variant
plot_pca.focus_point([263])
plot_pca.focus_point(263)
plot_pca.fit_view_by_ids([263])

In [ ]:
umap = UMAP(random_state=1)
embeddings_umap = umap.fit_transform(embeddings)
embeddings_umap = pd.DataFrame({'umap0': embeddings_umap[:, 0], 'umap1': embeddings_umap[:, 1]})
embeddings_umap = pd.concat((points, embeddings_umap), axis=1)
embeddings_umap

In [ ]:
plot_umap = cosmo(
    points = embeddings_umap,
    point_label_by = 'index',
    point_color_by = 'score',
    point_x_by = 'umap0',
    point_y_by = 'umap1',
    #point_cluster_by = 'cluster_id',
    #point_cluster_strength_by = 'cluster_strength',
    #point_include_columns=['CSQ_CLINVAR_CLNREVSTAT'],
    point_size=10,
    point_size_scale=1,
    simulation_repulsion=20, # Make sure unconnected points are pushed away, 0.14 good
    #simulation_link_spring=0.01,  # cluster similar points
    # repulsion 1 and spring 0.01 is good
    #simulation_link_distance=10,  # Affects initial positioning of linked nodes
    show_dynamic_labels = True,
    show_labels = True,
    render_links = False,
    #simulation_gravity = 10.0,
    #simulation_center = 0.8,
    random_seed = 0,
    simulation_decay=1000,  # Smaller is faster
    simulation_friction=0.8,
   #link_arrows = True
    disable_simulation=True
)
plot_umap

In [ ]:
plot_umap.focus_point([263])
plot_umap.focus_point(263)
plot_umap.fit_view_by_ids([263])

In [ ]:
# Cluster the explanations
umap_clusterer = UMAP(random_state=1, n_components=1)
explanations_cluster_umap = umap_clusterer.fit_transform(explanations)
explanations_cluster_umap = pd.DataFrame({'umap_cluster_0': explanations_clustered[:, 0].astype(int)})
explanations_cluster_umap_pca = pd.concat((embeddings_pca, explanations_cluster_umap), axis=1)
explanations_cluster_umap_umap = pd.concat((embeddings_umap, explanations_cluster_umap), axis=1)
explanations_cluster_umap_umap

In [ ]:
"""
For no movemment in plot, set
    simulation_repulsion=0
    simulation_gravity = 0.0,
    simulation_center = 0.0,
"""

plot_umap_explanations = cosmo(
    #points = explanations_cluster_umap_pca,
    points = explanations_cluster_umap_umap,
    point_label_by = 'index',
    #point_label_by = 'umap_cluster_0',
    point_color_by = 'score',
    point_cluster_by = 'umap_cluster_0',
    # PCA embeddings
    #point_x_by = 'pca0',
    #point_y_by = 'pca1',
    # UMAP embeddings
    point_x_by = 'umap0',
    point_y_by = 'umap1',
    point_size=10,
    point_size_scale=0.5,
    #links = links_embeddings,
    simulation_repulsion=0.0, # Make sure unconnected points are pushed away, 0.14 good
    #simulation_link_spring=0.01,  # cluster similar points
    # repulsion 1 and spring 0.01 is good
    #simulation_link_distance=10,  # Affects initial positioning of linked nodes
    show_dynamic_labels = True,
    show_labels = True,
    render_links = False,
    simulation_gravity = 0.01,
    simulation_center = 0.0,
    random_seed = 0,
    simulation_decay=1000,  # Smaller is faster
    simulation_friction=1.8,
   #link_arrows = True
    disable_simulation=False,
    simulation_cluster=0.01
)
plot_umap_explanations

In [ ]:
plot_umap_explanations.focus_point([263])
plot_umap_explanations.focus_point(263)
plot_umap_explanations.fit_view_by_ids([263])

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(2,1,1)
ax.scatter(explanations_cluster_umap_umap.index,
           explanations_cluster_umap_umap.umap_cluster_0,
          marker='.')
ax.scatter(263,
           explanations_cluster_umap_umap.iloc[263].umap_cluster_0,
           marker='.',
          color='r')
ax.set_ylabel('cluster ID')
ax.set_xlabel('variantID')
ax.grid('both')
ax = fig.add_subplot(2,1,2)
ax.scatter(explanations_cluster_umap_umap.index, explanations.CSQ_PolyPhen, marker='.')
ax.scatter(263, explanations.iloc[263].CSQ_PolyPhen,
           marker='.',
          color='r')
ax.set_ylabel('PolyPhen')

In [ ]:
umap_explanations = UMAP(random_state=1, n_components=2)
exp_2d_umap = umap_explanations.fit_transform(explanations)
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.scatter(exp_2d_umap[:, 0], exp_2d_umap[:, 1], marker='.')
ax.scatter(exp_2d_umap[263, 0],
           exp_2d_umap[263, 1],
           explanations.CSQ_PolyPhen.values,
           marker='.', color='red')

# Interactive clustering of Embeddings

FIXME: Broken in cosmo, requires updating to v0.0.47

In [ ]:
def setup_interactive_ui_handler(plot):
    """
    Cosmos UI handler for interacting with points
    """
    def ui_handler(event: dict):
        #print(event)
        cosmograph = event['owner']
        print(cosmograph)
        print(type(cosmograph))
        #print(vars(cosmograph))
        #cosmograph.disable_simulation = !cosmograph.disable_simulation
        return
        if event['name'] != 'clicked_point_index':
            return
        point_index = event['new']
        print(point_index)
        plot.select_point_by_index([point_index])
        plot.focus_point_by_index(point_index)
    
    plot.observe(ui_handler)

In [ ]:
"""
For no movemment in plot, set
    simulation_repulsion=0
    simulation_gravity = 0.0,
    simulation_center = 0.0,
"""

interactive = cosmo(
    #points = explanations_cluster_umap_pca,
    points = explanations_cluster_umap_umap,
    point_label_by = 'index',
    #point_label_by = 'umap_cluster_0',
    point_color_by = 'score',
    point_cluster_by = 'umap_cluster_0',
    # PCA embeddings
    #point_x_by = 'pca0',
    #point_y_by = 'pca1',
    # UMAP embeddings
    point_x_by = 'umap0',
    point_y_by = 'umap1',
    point_size=10,
    point_size_scale=0.5,
    #links = links_embeddings,
    simulation_repulsion=0.0, # Make sure unconnected points are pushed away, 0.14 good
    #simulation_link_spring=0.01,  # cluster similar points
    # repulsion 1 and spring 0.01 is good
    #simulation_link_distance=10,  # Affects initial positioning of linked nodes
    show_dynamic_labels = True,
    show_labels = True,
    render_links = False,
    simulation_gravity = 0.01,
    simulation_center = 0.0,
    random_seed = 0,
    simulation_decay=1000,  # Smaller is faster
    simulation_friction=1.8,
   #link_arrows = True
    disable_simulation=False,
    simulation_cluster=0.01
)
interactive

In [ ]:
setup_interactive_ui_handler(interactive)

# Plot model embeddings over explanability category
- Conservation vs functional effects 'CSQ_LoFtool', 'CSQ_PolyPhen'

Create a projection for embeddings thats based on dot projected explanation columns.

In [ ]:
expl_columns = explanations.columns
expl_columns

In [ ]:
expl_conservation_columns = ['CSQ_GERP++_RS', 'CSQ_phastCons100way_vertebrate', 'CSQ_phyloP100way_vertebrate']
expl_conservation = np.zeros_like(expl_columns)
for i, name in enumerate(expl_columns):
    if name in expl_conservation_columns:
        expl_conservation[i] = 1.0
expl_conservation

In [ ]:
expl_functional_columns = ['CSQ_LoFtool', 'CSQ_PolyPhen']
expl_functional = np.zeros_like(expl_columns)
for i, name in enumerate(expl_columns):
    if name in expl_functional_columns:
        expl_functional[i] = 1.0
expl_functional 

In [ ]:
df.columns

In [ ]:
v = np.dot(expl_functional, explanations.iloc[0])
v, explanations.iloc[0]

In [ ]:
ax_functional = []
ax_conservation = []
ax_score = []
for i in range(len(df)):
    ax_functional.append(np.dot(expl_functional, explanations.iloc[i]))
    ax_conservation.append(np.dot(expl_conservation, explanations.iloc[i]))
    ax_score.append(df.iloc[i].score)
df_expl_score = pd.DataFrame({'score': ax_score, 'cons': ax_conservation, 'func': ax_functional, 'index': df['index']})

In [ ]:
plot_expl = cosmo(
    #points = explanations_cluster_umap_pca,
    points = df_expl_score,
    point_label_by = 'index',
    point_color_by = 'score',
    # PCA embeddings
    #point_x_by = 'pca0',
    #point_y_by = 'pca1',
    # UMAP embeddings
    point_x_by = 'cons',
    point_y_by = 'func',
    point_size=10,
    point_size_scale=1.0,
    #links = links_embeddings,
    simulation_repulsion=0.0, # Make sure unconnected points are pushed away, 0.14 good
    #simulation_link_spring=0.01,  # cluster similar points
    # repulsion 1 and spring 0.01 is good
    #simulation_link_distance=10,  # Affects initial positioning of linked nodes
    show_dynamic_labels = True,
    show_labels = False,
    render_links = False,
    simulation_gravity = 0.0,
    simulation_center = 0.0,
    random_seed = 0,
    simulation_decay=1000,  # Smaller is faster
    simulation_friction=1.8,
   #link_arrows = True
    disable_simulation=False,
    simulation_cluster=0.01
)
plot_expl

In [ ]:
plot_expl.focus_point([263])
plot_expl.focus_point(263)
plot_expl.fit_view_by_ids([263])